# 7 — Hyperparameter Tuning v1 (Broad Search)

First Optuna hyperparameter search over the `OptunaModel` architecture. The search space is intentionally wide, covering both training hyperparameters and architecture choices simultaneously.

**Tuned parameters:** `learning_rate` [1e-5, 1e-2], `weight_decay` [1e-5, 1e-2], `dropout` [0, 0.5], `mixup_alpha` [0, 0.6], `n_blocks` {2, 3}, `kernel_size` {1, 3, 5}, `hidden_dims` (per-layer, 16–256). Models exceeding 1.5M parameters are pruned early. To limit search time.

**Setup:** TPE sampler (`seed=273`), MedianPruner, 30 epochs per trial, early stopping patience=5, stored in `optuna_brain.db` / study `brain_optuna_v1`. Includes a post-analysis cell that reconstructs approximate parameter counts per trial to identify efficient (low-param, low-loss) candidates.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

import optuna, optuna_dashboard


from src.dataset import *
from src.lightning import *
from src.models import *
from src.params import *
from src.utils import *
from optuna_integration import PyTorchLightningPruningCallback


## Setup: data & datamodule

In [2]:
files_dir = PROCESSED_DIR / "cropped" / "files"
metadata = pd.read_csv(DATA_DIR / "train.csv")
dm = BrainDataModule(metadata=metadata, spec_dir= files_dir, batch_size= 32, num_workers= 8, verbose= False)

## Objective function

In [ ]:
def objective(trial, datamodule):

    # def hyperparams to tune

    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log= True)
    dropout =   trial.suggest_float("dropout", 0, 0.5)
    mixup_alpha = trial.suggest_float("mixup_alpha", 0, 0.6)
    n_blocks = trial.suggest_int("n_blocks", 2, 3)
    kernel_size = trial.suggest_categorical("kernel_size",[1,3,5])
    hidden_dims  = [trial.suggest_int(f"dim_{i}",16, 256) for i in range(n_blocks*2)]


    # def model
    optuna_model = OptunaModel(n_channels= 4, n_classes= 6,
                               hidden_dims= hidden_dims,
                               kernel_size= kernel_size,
                               dropout= dropout)

    n_params = sum(p.numel() for p in optuna_model.parameters())
    if n_params > 1_500_000:
        raise optuna.TrialPruned()

    # def lightning module
    lit_optuna = BrainLightning(model= optuna_model, n_classes= 6,
                                lr= learning_rate, mixup= True, mixup_alpha= mixup_alpha,
                               scheduler= True, t_max= 30, weight_decay= weight_decay, verbose= False)

    # def callbacks adapted to tuning (short)
    callbacks = [
    ModelCheckpoint(
        dirpath=f"checkpoints/optuna/{trial.number}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
        filename="{epoch:02d}-{val_loss:.3f}",
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        min_delta=1e-3,
    ),
    PyTorchLightningPruningCallback(trial, monitor= "val_loss")
]



    trainer = pl.Trainer(
        max_epochs=30,
        accelerator="auto",
        devices="auto",
        callbacks=callbacks,
        logger= False,
        precision="bf16-mixed",
        enable_progress_bar= False, #silent
        log_every_n_steps=10,
    )

    trainer.fit(lit_optuna, datamodule= datamodule)

    val_loss_final = callbacks[0].best_model_score.item()
    return val_loss_final

## Launch study

In [4]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_brain.db")
study = optuna.create_study(
    direction = "minimize",
    sampler= optuna.samplers.TPESampler(seed= 273),
    pruner= optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=3 ),
    storage= storage,
    study_name= "brain_optuna_v1",
    load_if_exists= True
)

study.optimize(lambda trial: objective(trial, dm), n_trials= 30, show_progress_bar= True)

print(f"Best score : {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-04-11 13:06:58,539] Using an existing study with name 'brain_optuna_v1' instead of creating a new one.


  0%|          | 0/30 [00:00<?, ?it/s]

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[setup] Loading splits from cache: /Users/pablorougerie/code/Projets/brain_waves/data/cache/5_at_seed_273_min_vote_1.json
[setup] Cache loaded — 5 folds found


/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/pablorougerie/code/Projets/brain_waves/notebooks/checkpoints/optuna/1 exists and is not empty.
/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type        ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ OptunaModel │  1.0 M │ train │     0 │
│ 1 │ criterion │ KLDivLoss   │      0 │ train │     0 │
└───┴───────────┴─────────────┴────────┴───────┴───────┘

Trainable params: 1.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.0 M                                                                                                
Total estimated model params size (MB): 4                                                                          
Modules in train mode: 48                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/pablorougerie/code/Projets/brain_waves/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 000 | val_loss:   1.3981
Epoch 000 | val_loss:   1.0792
Epoch 001 | val_loss:   0.8522
Epoch 002 | val_loss:   0.8393
Epoch 003 | val_loss:   0.8193
Epoch 004 | val_loss:   0.8201
Epoch 005 | val_loss:   0.8162
Epoch 006 | val_loss:   0.7964
Epoch 007 | val_loss:   0.8074
Epoch 008 | val_loss:   0.7807
Epoch 009 | val_loss:   0.7538
Epoch 010 | val_loss:   0.7524
Epoch 011 | val_loss:   0.7288
Epoch 012 | val_loss:   0.7292
Epoch 013 | val_loss:   0.7131
Epoch 014 | val_loss:   0.7240
Epoch 015 | val_loss:   0.7313
Epoch 016 | val_loss:   0.7331
Epoch 017 | val_loss:   0.7205
Epoch 018 | val_loss:   0.7157
[I 2026-04-11 22:05:08,102] Trial 30 finished with value: 0.7131103277206421 and parameters: {'learning_rate': 0.0005973261636649988, 'weight_decay': 6.256260761848886e-05, 'dropout': 0.006340146974636765, 'mixup_alpha': 0.35332363272712314, 'n_blocks': 3, 'kernel_size': 5, 'dim_0': 33, 'dim_1': 38, 'dim_2': 178, 'dim_3': 74, 'dim_4': 60, 'dim_5': 250}. Best is trial 24 with va

## Visualization & parameter importance analysis

In [5]:
import plotly

In [6]:
optuna.visualization.plot_optimization_history(study)

In [7]:
optuna.visualization.plot_param_importances(study)


In [8]:
optuna.visualization.plot_parallel_coordinate(study)

## Best trial results

In [9]:
best_trial = study.best_trial

print(f"Best trial number : {best_trial.number}")
print(f"Best score        : {best_trial.value:.4f}")
print(f"Best params       : {best_trial.params}")

Best trial number : 24
Best score        : 0.6762
Best params       : {'learning_rate': 3.819779982915084e-05, 'weight_decay': 1.2034009705748483e-05, 'dropout': 0.08251441688664851, 'mixup_alpha': 0.3298238267268675, 'n_blocks': 3, 'kernel_size': 5, 'dim_0': 205, 'dim_1': 141, 'dim_2': 229, 'dim_3': 102, 'dim_4': 206, 'dim_5': 159}


## Save best trial to disk

In [10]:
import json

best_info = {
    "trial_number": best_trial.number,
    "score": best_trial.value,
    "params": best_trial.params
}

with open("checkpoints/optuna/best_trial.json", "w") as f:
    json.dump(best_info, f, indent=2)

## Bonus: efficiency analysis — params vs score

In [ ]:
import optuna
import numpy as np

study = optuna.load_study(
    study_name="brain_optuna_v1",  # adapte le nom
    storage="sqlite:///optuna_brain.db"
)

def count_params_from_trial(trial):
    # Reconstitue le nombre de params approximatif
    dims = [
        trial.params.get(f"dim_{i}", 64)
        for i in range(trial.params.get("n_blocks", 3) + 1)
    ]
    # Approximation : somme des produits consécutifs × kernel²
    k = trial.params.get("kernel_size", 3)
    total = 0
    in_ch = 4
    for d in dims[:-1]:
        total += in_ch * d * k * k
        in_ch = d
    return total

results = []
for trial in study.trials:
    if trial.state.name != "COMPLETE":
        continue
    score = trial.value
    try:
        n_params = count_params_from_trial(trial)
        results.append({
            "number": trial.number,
            "score": score,
            "n_params_approx": n_params,
            "params": trial.params
        })
    except:
        pass

# Trier par score croissant, filtrer les "petits" modèles
results.sort(key=lambda x: x["score"])
for r in results[:10]:
    print(f"Trial {r['number']:3d} | score={r['score']:.4f} | params_approx={r['n_params_approx']:,} | dims={[r['params'].get(f'dim_{i}') for i in range(5)]}")

Trial  24 | score=0.6762 | params_approx=1,550,350 | dims=[205, 141, 229, 102, 206]
Trial  27 | score=0.6798 | params_approx=235,950 | dims=[39, 42, 182, 43, 20]
Trial  26 | score=0.6829 | params_approx=241,675 | dims=[41, 43, 180, 94, 74]
Trial   7 | score=0.6863 | params_approx=650,525 | dims=[47, 109, 190, 93, 184]
Trial  15 | score=0.6900 | params_approx=246,537 | dims=[127, 95, 156, 107, 109]
Trial  25 | score=0.6961 | params_approx=481,806 | dims=[207, 146, 154, 59, 225]
Trial  22 | score=0.6971 | params_approx=1,420,450 | dims=[175, 141, 223, 115, 193]
Trial  21 | score=0.6979 | params_approx=973,000 | dims=[175, 105, 189, 165, 145]
Trial  20 | score=0.7027 | params_approx=962,200 | dims=[172, 105, 188, 161, 149]
Trial  23 | score=0.7040 | params_approx=1,638,800 | dims=[116, 192, 223, 116, 206]


## Conclusion

**Best trial: #24, val_loss = 0.6762**

Best params:
```
learning_rate = 3.82e-5   weight_decay = 1.20e-5   dropout = 0.083
mixup_alpha   = 0.330     n_blocks = 3             kernel_size = 5
dims = [205, 141, 229, 102, 206, 159]   (~1.55M params, near the 1.5M cap)
```

**Parameter importance analysis findings:**
- `weight_decay`, `dropout`, `mixup_alpha` → low importance: model is robust to these values → fixed in v2
- `kernel_size` and `learning_rate` are the most important. kernel_size being at 5, we don't know if 7 wouldn't be even better. 
- `kernel_size = 5` is strongly preferred over 1 and 3 across all top trials
- `n_blocks = 3` is consistent across best trials → fixed in v2

**Efficiency analysis** (params vs score): trial 27 achieves val_loss=0.6798 with only ~236K parameters (vs 1.55M for trial 24). This shows that a small, well-tuned model nearly matches the overparameterized best — a key insight for the pruning stage. So trial 27 will probably be preferred over 24. 

**Next:** run a refined search fixing the low-importance parameters and focusing on `learning_rate`, `kernel_size`, and `hidden_dims` within a tighter, efficiency-aware budget (`8_optuna_v2`).